# baseline v3 (로컬 GPU 버전)

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

원래 Colab(T4 GPU)용으로 개발된 코드를 **로컬 Windows PC의 NVIDIA GPU**에서 실행되도록 수정했습니다.

#### Colab 버전과 달라진 점
| 항목 | Colab 원본 | 로컬 버전 |
|---|---|---|
| PyTorch | CUDA 12.1 빌드 (`cu121`) | CUDA 12.8 빌드 (`cu128`) — RTX 50 시리즈(Blackwell, sm_120)는 cu128 이상이 필요 |
| 패키지 설치 | `!pip` | `%pip` (현재 커널의 가상환경에 설치) |
| 데이터 위치 | 구글 드라이브 마운트 | 로컬 폴더 경로 (`DATA_PATH`) |
| 정밀도 | fp16 양자화 연산 + bf16 autocast 혼용 | GPU가 지원하면 bf16으로 통일, GradScaler는 fp16일 때만 사용 |
| 모델 배치 | `device_map="auto"` + `.to(device)` | `device_map={"": 0}` (GPU 0번에 전부 올림, CPU 오프로딩 방지) |
| 저장 경로 | `/content/...` | `./outputs/...` |

#### 실행 전 준비
- VS Code 우측 상단 커널 선택에서 이 프로젝트의 `.venv` (Python 3.13)를 선택하세요.
- NVIDIA 드라이버가 설치되어 있어야 합니다. (터미널에서 `nvidia-smi`로 확인)

# 환경 준비

로컬 가상환경(`.venv`)에 필요한 라이브러리를 설치합니다. **처음 한 번만** 실행하면 됩니다.

- 아래 셀 실행 (PyTorch 다운로드 약 3GB)
- 실행 완료 후 VS Code 노트북 상단의 **Restart(커널 다시 시작)** 클릭
- `transformers`는 GitHub 최신 소스에서 설치하므로 `git`이 필요합니다.

In [1]:
# RTX 50 시리즈(Blackwell)는 CUDA 12.8 이상으로 빌드된 PyTorch가 필요합니다.
%pip install -q --index-url https://download.pytorch.org/whl/cu128 torch torchvision
%pip install -q git+https://github.com/huggingface/transformers accelerate
# bitsandbytes는 0.46 이상부터 Windows + CUDA 12.8 휠을 제공합니다.
# ipywidgets는 노트북에서 tqdm 진행바를 표시하는 데 필요합니다.
%pip install -q "peft>=0.13.2" "bitsandbytes>=0.46.1" datasets pillow pandas ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# GPU 환경 점검
import torch

print("Torch version :", torch.__version__)
print("CUDA (torch)  :", torch.version.cuda)
print("cuDNN version :", torch.backends.cudnn.version())
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    print("GPU           :", props.name)
    print("VRAM          :", f"{props.total_memory / 1024**3:.1f} GB")
    print("Compute cap.  :", f"sm_{major}{minor}")
    print("bf16 지원     :", torch.cuda.is_bf16_supported())

    # 설치된 PyTorch가 이 GPU 아키텍처용 커널을 포함하는지 확인
    if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        print("\n[경고] 설치된 PyTorch가 이 GPU를 지원하지 않습니다. 위 설치 셀의 cu128 빌드를 확인하세요.")

    # 간단한 연산 테스트
    x = torch.randn(1024, 1024, device="cuda")
    print("GPU 연산 테스트:", "OK" if torch.isfinite(x @ x).all().item() else "FAIL")
else:
    print("\n[경고] GPU를 찾지 못했습니다. 드라이버(nvidia-smi)와 PyTorch CUDA 빌드를 확인하세요.")

Torch version : 2.11.0+cu128
CUDA (torch)  : 12.8
cuDNN version : 91900
CUDA 사용 가능: True
GPU           : NVIDIA GeForce RTX 5060 Ti
VRAM          : 15.9 GB
Compute cap.  : sm_120
bf16 지원     : True
GPU 연산 테스트: OK


# 데이터 준비

개발에 필요한 데이터를 로컬 폴더에 준비합니다. 기본 경로는 노트북과 같은 폴더 아래의 `data` 폴더입니다.

```
mosquito_AI/
├─ (260908)_baseline_local.ipynb
└─ data/
   ├─ train.csv   (id, path, question, a, b, c, d, answer)
   ├─ dev.csv
   ├─ test.csv    (id, path, question, a, b, c, d)
   └─ (이미지 파일들 — csv의 path 열은 data 폴더 기준 상대경로)
```

다른 위치에 데이터가 있다면 아래 셀의 `DATA_PATH`만 바꾸면 됩니다.
Colab의 구글 드라이브 마운트 셀은 로컬에서는 필요 없으므로 삭제했습니다.

# 라이브러리, 데이터, 설정

In [3]:
import os, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda:0")
SEED = 42
random.seed(SEED); np.random.seed(SEED); 
torch.manual_seed(SEED); torch.cuda.manual_seed(SEED); 
torch.backends.cudnn.deterministic = True; 
torch.backends.cudnn.benchmark = False

DATA_PATH  = Path.cwd() / "data"
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

labels = pd.read_csv(DATA_PATH / "train_labels.csv")      # id, x, y, z (+80ms 정답)
sub    = pd.read_csv(DATA_PATH / "sample_submission.csv")  # id, x, y, z

def load_tracks(folder, ids):
    # 파일 하나 = 11개 시점 (timestep_ms -400 ~ 0)
    tracks = []
    for i in tqdm(ids, desc=folder.name):
        df = pd.read_csv(folder / f"{i}.csv").sort_values("timestep_ms")
        tracks.append(df[["x", "y", "z"]].to_numpy())
    return np.stack(tracks).astype(np.float32)

X_train = load_tracks(DATA_PATH / "train", labels["id"])   # (10000, 11, 3)
Y_train = labels[["x", "y", "z"]].to_numpy(np.float32)     # (10000, 3)
X_test  = load_tracks(DATA_PATH / "test", sub["id"])       # (10000, 11, 3)


train:   0%|          | 0/10000 [00:00<?, ?it/s]

test:   0%|          | 0/10000 [00:00<?, ?it/s]

## 입력 특징과 정답 만들기
학습 데이터와 평가 데이터는 장소가 다릅니다. 그래서 절대 좌표 대신 이동량을 입력으로 씁니다. 미터 단위는 값이 너무 작아서 100을 곱해 센티미터 단위로 학습합니다.

In [12]:
SCALE   = 100.0   # m -> cm
N_STEPS = 10      # 관측 11개 → 이동량 10개

def baseline(X):
    # 등속 외삽: 80ms 뒤 = 마지막 위치 + 마지막 40ms 이동량 x 2
    return X[:, -1] + 2 * (X[:, -1] - X[:, -2])

def to_features(vel, n_valid):
    """vel: (N, 10, 3) m 단위 이동량, n_valid: (N,) 진짜 이동량 개수
       → (N, 10, 4): 이동량 x, y, z (cm) + 진짜 이동량이면 1, 채운 자리면 0"""
    mask = (np.arange(N_STEPS)[None, :] >= N_STEPS - n_valid[:, None]).astype(np.float32)
    return np.concatenate([vel * SCALE, mask[..., None]], axis=2).astype(np.float32)

def full_features(X):
    """관측 11개가 모두 있는 궤적 (원래 학습 샘플, 테스트)"""
    return to_features(np.diff(X, axis=1), np.full(len(X), N_STEPS))

# ---- 여러 시점 동시 예측용 ----
HORIZONS  = [1, 2, 3]          # 1칸(40ms), 2칸(80ms), 3칸(120ms) 뒤
H_WEIGHTS = [0.5, 1.0, 0.5]    # 손실에서 각 시점의 비중
MAIN      = HORIZONS.index(2)  # 제출에 쓰는 80ms 시점의 위치

def build_windows(X, Y, min_win=3):
    """학습 궤적에서만 구간을 만듦. 구간 마지막 인덱스 e에서 두 칸(80ms) 뒤가 정답.
       e = min_win-1 ... 8 → 정답은 관측값 X[:, e+2]
       e = 10              → 정답은 라벨 Y (원래 샘플)"""
    feats, resids, tracks = [], [], []
    N = len(X)
    for e in list(range(min_win - 1, 9)) + [10]:
        hist   = X[:, :e + 1]
        target = Y if e == 10 else X[:, e + 2]
        vel    = np.diff(hist, axis=1)                                         # (N, e, 3)
        vel    = np.concatenate([np.zeros((N, N_STEPS - e, 3), np.float32), vel], axis=1)  # 앞쪽을 0으로 채움
        feats.append(to_features(vel, np.full(N, e)))
        resids.append(((target - baseline(hist)) * SCALE).astype(np.float32))
        tracks.append(np.arange(N))
    return np.concatenate(feats), np.concatenate(resids), np.concatenate(tracks)

feats_train = full_features(X_train)                   # (10000, 10, 4) 원래 샘플
resid_train = (Y_train - baseline(X_train)) * SCALE   # (10000, 3)
feats_test  = full_features(X_test)                    # (10000, 10, 4)



## Dataset
z축을 기준으로 무작위 회전시키는 증강이 일반화에 가장 효과가 컸습니다. 입력과 정답을 반드시 같은 각도로 함께 돌려야 합니다. z축은 위쪽 방향이라 회전하지 않습니다.

In [13]:
class GPULoader:
    """데이터를 GPU에 올려 두고 배치째 회전 증강. train_stage의 `for v, t in loader`와 그대로 호환."""
    def __init__(self, feats, targets, batch_size, shuffle=False, augment=False):
        self.f = torch.from_numpy(feats).to(device)
        self.t = torch.from_numpy(targets).to(device)
        self.bs, self.shuffle, self.augment = batch_size, shuffle, augment

    def __len__(self):
        return math.ceil(len(self.f) / self.bs)

    def __iter__(self):
        n = len(self.f)
        order = torch.randperm(n, device=device) if self.shuffle else torch.arange(n, device=device)
        for b in range(0, n, self.bs):
            j = order[b:b + self.bs]
            f, t = self.f[j], self.t[j]
            if self.augment:
                th = torch.rand(len(j), device=device) * 2 * math.pi
                c, s = torch.cos(th), torch.sin(th)
                z, o = torch.zeros_like(th), torch.ones_like(th)
                R = torch.stack([torch.stack([c, -s, z], -1),
                                 torch.stack([s,  c, z], -1),
                                 torch.stack([z,  z, o], -1)], -2)            # (B, 3, 3) z축 회전
                f = torch.cat([torch.einsum("bij,btj->bti", R, f[..., :3]),  # 이동량만 회전
                               f[..., 3:]], dim=-1)                          # 네 번째 채널은 그대로
                if t.dim() == 2:      # 정답이 (B, 3): 기존 방식
                    t = torch.einsum("bij,bj->bi", R, t)
                else:                 # 정답이 (B, 3시점, 4): 보정값만 회전, 정답 여부는 그대로
                    t = torch.cat([torch.einsum("bij,bhj->bhi", R, t[..., :3]), t[..., 3:]], dim=-1)

            yield f.reshape(len(j), -1), t


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 처음 한 번만 받고, 이후에는 로컬 캐시(`%USERPROFILE%\.cache\huggingface`)에서 불러옵니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [14]:
class TrajMLP(nn.Module):
    def __init__(self, in_dim=40, hidden=256, depth=3, dropout=0.0, n_horizons=1):
        super().__init__()
        layers, d = [], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.GELU(), nn.Dropout(dropout)]
            d = hidden
        layers.append(nn.Linear(d, 3 * n_horizons))
        self.net = nn.Sequential(*layers)
        self.n_h = n_horizons

    def forward(self, x):
        out = self.net(x)
        if self.n_h == 1:
            return out
        out = out.view(len(x), self.n_h, 3)
        # 학습 중에는 모든 시점, 평가·예측 때는 80ms 시점만 돌려줌
        return out if self.training else out[:, MAIN]


# fine-tuning

#### 참고

    손실은 평가 기준과 같은 3D 유클리드 거리로 둡니다. 모델이 작아서 bf16, GradScaler, gradient 누적은 필요 없습니다.

In [43]:
MIN_WIN    = 3      # 구간 최소 길이 (관측 개수)
EPOCHS     = 60     # 데이터가 8배라서 200 → 50
FT_EPOCHS  = 15     # 같은 이유로 50 → 15 (시작값)
BATCH_SIZE = 256
N_FOLDS    = 5
USE_FT     = True
HIT_RADIUS = 0.01   # m
RUN_SEED = 42 
EXP    = "mh512"   # 실험 이름: 결과 파일 이름에 붙음
HIDDEN = 512        # 모델 폭
DEPTH  = 4          # 모델 층 수
USE_MH = True      # True면 여러 시점 동시 예측

# 샘플마다 0~4 중 하나의 fold 번호를 붙임 (fold마다 2000개)
perm = np.random.RandomState(RUN_SEED).permutation(len(X_train))
folds = np.empty(len(X_train), dtype=int)
folds[perm] = np.arange(len(X_train)) % N_FOLDS
print("fold별 샘플 수:", np.bincount(folds))


fold별 샘플 수: [2000 2000 2000 2000 2000]


## 함수

In [37]:

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def dist3d(pred, target):
    return torch.linalg.norm(pred - target, dim=1)

def hit_loss(pred, target, radius=1.0, tau=0.15, dist_weight=0.1):
    d = dist3d(pred, target)
    return (1 - torch.sigmoid((radius - d) / tau)).mean() + dist_weight * d.mean()

def evaluate(model, loader):
    model.eval()
    with torch.no_grad():
        d = torch.cat([dist3d(model(v.to(device)), t.to(device)) for v, t in loader]) / SCALE  # m
    return d.mean().item(), (d < HIT_RADIUS).float().mean().item()

HISTORY = {}   # 추가: (fold, 단계)별 기록

def train_stage(model, train_loader, valid_loader, loss_fn, epochs, lr, save_path, keep_best, desc):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    best, best_epoch, hist = float("inf"), epochs, []           # 추가
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        total, n = 0.0, 0                                       # 추가
        for v, t in train_loader:
            v, t = v.to(device), t.to(device)
            loss = loss_fn(model(v), t)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(v); n += len(v)         # 추가
        scheduler.step()
        val_dist, val_hit = evaluate(model, valid_loader)       # 매 epoch 검증
        hist.append((epoch + 1, total / n, val_dist * 100, val_hit * 100))   # 추가
        if keep_best and val_dist < best:
            best, best_epoch = val_dist, epoch + 1
            torch.save(model.state_dict(), save_path)
    if not keep_best:
        torch.save(model.state_dict(), save_path)
    HISTORY[desc] = (hist, best_epoch)                          # 추가
    model.load_state_dict(torch.load(save_path))
    return model

def predict(model, feats):
    """보정값 예측, cm 단위"""
    model.eval()
    with torch.no_grad():
        x = torch.from_numpy(feats.reshape(len(feats), -1)).to(device)
        return model(x).cpu().numpy()

def report(name, pred_cm, idx=None):
    idx = np.arange(len(resid_train)) if idx is None else idx
    d = np.linalg.norm(pred_cm - resid_train[idx], axis=1) / SCALE
    print(f"{name:12s} 평균 오차 {d.mean()*100:.3f}cm | 명중률 {(d < HIT_RADIUS).mean()*100:.2f}%")
    
HW = torch.tensor(H_WEIGHTS, device=device)

def mh_loss(pred, t):
    """pred (B, 3시점, 3), t (B, 3시점, 4) → 정답 있는 시점만 가중 평균한 3D 거리"""
    d = torch.linalg.norm(pred - t[..., :3], dim=2)
    w = t[..., 3] * HW
    return (d * w).sum() / w.sum()

def mh_hit_loss(pred, t, radius=1.0, tau=0.15, dist_weight=0.1):
    """80ms 시점은 명중률 손실, 전체 시점은 거리 손실을 조금 섞음"""
    d = torch.linalg.norm(pred - t[..., :3], dim=2)
    m = t[..., 3]
    main_m = m[:, MAIN]
    soft_miss = 1 - torch.sigmoid((radius - d[:, MAIN]) / tau)
    hit = (soft_miss * main_m).sum() / main_m.sum().clamp(min=1)
    aux = (d * m * HW).sum() / (m * HW).sum()
    return hit + dist_weight * aux



## 테스트 시 회전 평균
회전 증강으로 학습한 모델이라 어느 방향으로 돌려도 비슷하게 예측해야 합니다. 입력을 8방향으로 돌려 예측하고, 결과를 원래 방향으로 되돌려 평균합니다.

In [33]:
def predict_tta(model, feats, K=8):
    """입력을 K방향으로 돌려 예측하고 원래 방향으로 되돌려 평균 (cm 단위)"""
    model.eval()
    f = torch.from_numpy(feats).to(device)                     # (N, 10, 4)
    out = 0
    with torch.no_grad():
        for k in range(K):
            th = 2 * math.pi * k / K
            c, s = math.cos(th), math.sin(th)
            R = torch.tensor([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=torch.float32, device=device)
            fr = torch.cat([f[..., :3] @ R.T, f[..., 3:]], dim=-1)   # 이동량만 회전
            p = model(fr.reshape(len(fr), -1)) @ R                   # 예측을 원래 방향으로 되돌림
            out = out + p / K
    return out.cpu().numpy()


## K-fold 학습 셀

In [44]:
if USE_MH:
    w_feats, w_tgt, w_track = build_windows_mh(X_train, Y_train, MIN_WIN)
    loss_base, loss_ft, n_h = mh_loss, mh_hit_loss, len(HORIZONS)
else:
    w_feats, w_tgt, w_track = build_windows(X_train, Y_train, MIN_WIN)
    loss_base, loss_ft, n_h = (lambda p, t: dist3d(p, t).mean()), hit_loss, 1
print(f"[{EXP}] 학습 구간 수:", len(w_feats))
in_dim = feats_train.shape[1] * feats_train.shape[2]       # 10 × 4 = 40

oof_base  = np.zeros_like(resid_train)
oof_ft    = np.zeros_like(resid_train)
test_base = np.zeros((len(X_test), 3), dtype=np.float32)
test_ft   = np.zeros((len(X_test), 3), dtype=np.float32)

for k in range(N_FOLDS):
    set_seed(RUN_SEED + k)
    va_idx  = np.where(folds == k)[0]
    tr_mask = folds[w_track] != k     # 검증 궤적에서 나온 구간은 전부 학습에서 제외

    train_loader = GPULoader(w_feats[tr_mask], w_tgt[tr_mask], BATCH_SIZE, shuffle=True, augment=True)
    valid_loader = GPULoader(feats_train[va_idx], resid_train[va_idx], 4096)   # 검증은 원래 샘플만

    # 1단계: 거리 손실
    model = TrajMLP(in_dim=in_dim, hidden=HIDDEN, depth=DEPTH, n_horizons=n_h).to(device)
    model = train_stage(model, train_loader, valid_loader, loss_base, EPOCHS, 1e-3,
                        OUTPUT_DIR / f"fold{k}_{EXP}_s{RUN_SEED}_base.pt", keep_best=True, desc=f"fold{k} base")
    oof_base[va_idx] = predict_tta(model, feats_train[va_idx])
    test_base += predict_tta(model, feats_test) / N_FOLDS

    # 2단계: 명중률 손실로 추가 학습
    if USE_FT:
        model = train_stage(model, train_loader, valid_loader, loss_ft, FT_EPOCHS, 1e-4,
                            OUTPUT_DIR / f"fold{k}_{EXP}_s{RUN_SEED}_ft.pt", keep_best=False, desc=f"fold{k} ft")
        oof_ft[va_idx] = predict_tta(model, feats_train[va_idx])
        test_ft += predict_tta(model, feats_test) / N_FOLDS      # 버그 수정: predict → predict_tta

    print(f"--- fold {k} ---")
    report("base", oof_base[va_idx], va_idx)
    if USE_FT:
        report("ft",  oof_ft[va_idx], va_idx)
        report("avg", (oof_base[va_idx] + oof_ft[va_idx]) / 2, va_idx)


[mh512] 학습 구간 수: 90000


fold0 base:   0%|          | 0/60 [00:00<?, ?it/s]

fold0 ft:   0%|          | 0/15 [00:00<?, ?it/s]

--- fold 0 ---
base         평균 오차 1.115cm | 명중률 67.20%
ft           평균 오차 1.117cm | 명중률 68.00%
avg          평균 오차 1.114cm | 명중률 67.80%


fold1 base:   0%|          | 0/60 [00:00<?, ?it/s]

...

fold3 ft:   0%|          | 0/15 [00:00<?, ?it/s]

--- fold 3 ---
base         평균 오차 1.163cm | 명중률 66.60%
ft           평균 오차 1.170cm | 명중률 67.35%
avg          평균 오차 1.165cm | 명중률 67.20%


fold4 base:   0%|          | 0/60 [00:00<?, ?it/s]

fold4 ft:   0%|          | 0/15 [00:00<?, ?it/s]

--- fold 4 ---
base         평균 오차 1.154cm | 명중률 65.85%
ft           평균 오차 1.154cm | 명중률 66.35%
avg          평균 오차 1.152cm | 명중률 66.00%


### 자동 저장

In [45]:
tag = f"{EXP}_s{RUN_SEED}"
np.save(OUTPUT_DIR / f"test_base_{tag}.npy", test_base)
np.save(OUTPUT_DIR / f"test_ft_{tag}.npy",   test_ft)
np.save(OUTPUT_DIR / f"oof_base_{tag}.npy",  oof_base)
np.save(OUTPUT_DIR / f"oof_ft_{tag}.npy",    oof_ft)
print("저장 완료:", tag)


저장 완료: mh512_s42


In [46]:
hist, best_epoch = HISTORY["fold0 base"]
print(f"저장된 epoch: {best_epoch} / {len(hist)}")
for ep, tr_loss, v_dist, v_hit in hist[::5]:
    print(f"[{ep:3d}] 학습 손실 {tr_loss:.4f} | 검증 오차 {v_dist:.3f}cm | 검증 명중률 {v_hit:.2f}%")


저장된 epoch: 39 / 60
[  1] 학습 손실 1.3449 | 검증 오차 1.247cm | 검증 명중률 60.10%
[  6] 학습 손실 1.2565 | 검증 오차 1.174cm | 검증 명중률 64.95%
[ 11] 학습 손실 1.2401 | 검증 오차 1.157cm | 검증 명중률 65.90%
[ 16] 학습 손실 1.2285 | 검증 오차 1.149cm | 검증 명중률 65.70%
[ 21] 학습 손실 1.2205 | 검증 오차 1.150cm | 검증 명중률 66.50%
[ 26] 학습 손실 1.2103 | 검증 오차 1.137cm | 검증 명중률 66.60%
[ 31] 학습 손실 1.1999 | 검증 오차 1.138cm | 검증 명중률 66.35%
[ 36] 학습 손실 1.1897 | 검증 오차 1.137cm | 검증 명중률 65.90%
[ 41] 학습 손실 1.1787 | 검증 오차 1.133cm | 검증 명중률 66.35%
[ 46] 학습 손실 1.1687 | 검증 오차 1.138cm | 검증 명중률 65.80%
[ 51] 학습 손실 1.1615 | 검증 오차 1.131cm | 검증 명중률 65.75%
[ 56] 학습 손실 1.1590 | 검증 오차 1.131cm | 검증 명중률 66.05%


# inference

In [47]:
print("=== 전체 OOF (학습 데이터 10000개) ===")
report("등속 외삽", np.zeros_like(resid_train))
report("base", oof_base)
if USE_FT:
    report("ft",  oof_ft)
    report("avg", (oof_base + oof_ft) / 2)


=== 전체 OOF (학습 데이터 10000개) ===
등속 외삽        평균 오차 1.294cm | 명중률 57.87%
base         평균 오차 1.123cm | 명중률 66.90%
ft           평균 오차 1.126cm | 명중률 67.20%
avg          평균 오차 1.123cm | 명중률 67.15%


In [48]:
for exp in ["mlp256", "mlp512", "mh"]:
    tag = f"{exp}_s42"
    if (OUTPUT_DIR / f"oof_base_{tag}.npy").exists():
        oof = (np.load(OUTPUT_DIR / f"oof_base_{tag}.npy") + np.load(OUTPUT_DIR / f"oof_ft_{tag}.npy")) / 2
        report(exp, oof)


mlp256       평균 오차 1.138cm | 명중률 66.45%
mlp512       평균 오차 1.131cm | 명중률 66.76%
mh           평균 오차 1.129cm | 명중률 66.71%


In [49]:
resid_test = (test_base + test_ft) / 2 if USE_FT else test_base
sub[["x", "y", "z"]] = baseline(X_test) + resid_test / SCALE
sub.to_csv(OUTPUT_DIR / f"submission_{EXP}_s{RUN_SEED}.csv", index=False)
print(sub.head())


           id         x         y         z
0  TEST_00001  3.989045 -1.052682  0.045463
1  TEST_00002  1.929827  0.618323  0.797606
2  TEST_00003  0.805910 -0.189844  0.146951
3  TEST_00004  3.526994 -1.645150 -0.941940
4  TEST_00005  2.302130  0.016205 -0.040632


# (선택) GPU 메모리 정리

다른 실험을 이어서 하기 전에 VRAM을 비우고 싶을 때 실행합니다. 커널 재시작이 가장 확실한 방법입니다.

In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print(f"현재 VRAM 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

현재 VRAM 사용량: 0.03 GB
